Processamento de Imagens Coloridas - Códigos em Python Baseado no material da disciplina PDSI

Autor: edvonaldohoracio@gmail.com / https://github.com/NaldoHoracio

Este script contém implementações em Python/NumPy/OpenCV para os conceitos
de processamento de imagens coloridas vistos nos slides, incluindo:

1. Conversão de Modelos de Cores (RGB <-> HSI, RGB -> CMY, RGB -> CMYK)
2. Processamento em Pseudo-cores (Fatiamento por Intensidade)
3. Transformações de Cores (Complemento, Ajuste de Brilho/Intensidade)
4. Filtragem Espacial em Cores (Suavização e Aguçamento com Laplaciano)

In [ ]:
# Download do OpenCV
!pip install opencv-python

In [ ]:
# Importação das bibliotecas
import os
import cv2
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow

In [ ]:
# Define o backend da biblioteca Matplotlib para o modo Agg (Anti-Grain Geometry).
# Esse backend gera gráficos exclusivamente em memória e em arquivos de imagem, sem abrir nenhuma janela na tela.
# matplotlib.use('Agg')
#%matplotlib# Reativa a exibição de imagens no google colab

In [ ]:
# Download do repositório
if os.path.isdir("pdsi"):
  print("Diretório já existe!")
else:
  !git clone https://github.com/NaldoHoracio/pdsi

# CONVERSÃO DE MODELOS DE CORES

In [ ]:
def rgb_to_hsi(rgb):
    """
    Converte uma imagem RGB (com valores no intervalo [0, 1]) para o espaço HSI.
    Fórmulas matemáticas exatas do Slide 20:
    - I = 1/3 * (R + G + B)
    - S = 1 - (3 / (R+G+B)) * min(R, G, B)
    - H obtido através do cosseno inverso (theta) e verificação de B <= G.

    Retorna: imagem HSI (H em graus [0, 360], S em [0, 1], I em [0, 1])
    """
    # Evita erros de float32
    rgb = np.clip(rgb, 0.0, 1.0).astype(np.float32)
    R = rgb[:, :, 0]
    G = rgb[:, :, 1]
    B = rgb[:, :, 2]

    # Intensidade
    I = (R + G + B) / 3.0

    # Saturação
    sum_rgb = R + G + B
    sum_rgb_safe = np.where(sum_rgb == 0, 1e-5, sum_rgb)
    min_rgb = np.minimum(np.minimum(R, G), B)
    S = 1.0 - (3.0 / sum_rgb_safe) * min_rgb
    S = np.clip(S, 0.0, 1.0)

    # Matiz (Hue)
    num = 0.5 * ((R - G) + (R - B))
    den = np.sqrt((R - G)**2 + (R - B)*(G - B))
    den_safe = np.where(den == 0, 1e-5, den)

    val = num / den_safe
    val = np.clip(val, -1.0, 1.0)  # Garante limites do acos
    theta = np.arccos(val)

    H = np.zeros_like(R)
    H[B <= G] = theta[B <= G]
    H[B > G] = 2.0 * np.pi - theta[B > G]

    # Convertemos de radianos para graus
    H_deg = np.degrees(H)
    H_deg[S == 0] = 0.0  # Para cinzas puros, matiz é convencionado como 0

    return np.stack([H_deg, S, I], axis=-1)

In [ ]:
def hsi_to_rgb(hsi):
    """
    Converte uma imagem do espaço HSI de volta para RGB [0, 1].
    Implementação baseada nos setores RG, GB e BR (Slides 21 e 22).
    """
    H = np.mod(hsi[:, :, 0], 360.0)
    S = np.clip(hsi[:, :, 1], 0.0, 1.0)
    I = np.clip(hsi[:, :, 2], 0.0, 1.0)

    R = np.zeros_like(H)
    G = np.zeros_like(H)
    B = np.zeros_like(H)

    # Caso 1: Setor RG (0 <= H < 120 graus)
    idx1 = (H >= 0) & (H < 120)
    H1 = np.radians(H[idx1])
    I1 = I[idx1]
    S1 = S[idx1]
    B[idx1] = I1 * (1.0 - S1)
    R[idx1] = I1 * (1.0 + (S1 * np.cos(H1)) / np.cos(np.radians(60.0) - H1))
    G[idx1] = 3.0 * I1 - (R[idx1] + B[idx1])

    # Caso 2: Setor GB (120 <= H < 240 graus)
    idx2 = (H >= 120) & (H < 240)
    H2 = np.radians(H[idx2] - 120.0)
    I2 = I[idx2]
    S2 = S[idx2]
    R[idx2] = I2 * (1.0 - S2)
    G[idx2] = I2 * (1.0 + (S2 * np.cos(H2)) / np.cos(np.radians(60.0) - H2))
    B[idx2] = 3.0 * I2 - (R[idx2] + G[idx2])

    # Caso 3: Setor BR (240 <= H < 360 graus)
    idx3 = (H >= 240) & (H < 360)
    H3 = np.radians(H[idx3] - 240.0)
    I3 = I[idx3]
    S3 = S[idx3]
    G[idx3] = I3 * (1.0 - S3)
    B[idx3] = I3 * (1.0 + (S3 * np.cos(H3)) / np.cos(np.radians(60.0) - H3))
    R[idx3] = 3.0 * I3 - (G[idx3] + B[idx3])

    rgb = np.stack([R, G, B], axis=-1)
    return np.clip(rgb, 0.0, 1.0)

In [ ]:
def rgb_to_cmy(rgb):
    """
    Converte uma imagem RGB [0, 1] para CMY [0, 1].
    C = 1 - R, M = 1 - G, Y = 1 - B (Slide 16)
    """
    return 1.0 - rgb

In [ ]:
def rgb_to_cmyk(rgb):
    """
    Converte uma imagem RGB [0, 1] para CMYK [0, 1].
    Adiciona o componente preto (K) para melhor impressão (Slide 16).
    """
    C = 1.0 - rgb[:, :, 0]
    M = 1.0 - rgb[:, :, 1]
    Y = 1.0 - rgb[:, :, 2]
    K = np.minimum(np.minimum(C, M), Y)

    one_minus_k = 1.0 - K
    one_minus_k_safe = np.where(one_minus_k == 0, 1e-5, one_minus_k)

    C_out = (C - K) / one_minus_k_safe
    M_out = (M - K) / one_minus_k_safe
    Y_out = (Y - K) / one_minus_k_safe

    C_out[K == 1.0] = 0.0
    M_out[K == 1.0] = 0.0
    Y_out[K == 1.0] = 0.0

    return np.stack([C_out, M_out, Y_out, K], axis=-1)

# PROCESSAMENTO EM PSEUDO-CORES

In [ ]:
def fatiamento_intensidade(gray, limiares, cores):
    """
    Método de Fatiamento por Intensidade (Slide 28).
    Transforma uma imagem em tons de cinza [0, 1] em uma imagem colorida (pseudo-cores)
    atribuindo cores específicas a faixas de valores delimitadas por limiares.
    """
    h, w = gray.shape
    out = np.zeros((h, w, 3), dtype=np.float32)

    # Constrói a lista de limites incluindo o mínimo e máximo possíveis
    limiares_completos = [0.0] + list(limiares) + [1.0]
    for i in range(len(limiares_completos) - 1):
        min_val = limiares_completos[i]
        max_val = limiares_completos[i+1]

        if i == len(limiares_completos) - 2:
            mask = (gray >= min_val) & (gray <= max_val)
        else:
            mask = (gray >= min_val) & (gray < max_val)

        out[mask] = cores[i]

    return out

# TRANSFORMAÇÕES DE CORES

In [ ]:
def complemento_cor(rgb):
    """
    Calcula o complemento de cor de uma imagem RGB [0, 1] (Slide 40).
    Equivale ao oposto no círculo de cores (Slide 40).
    """
    return 1.0 - rgb

In [ ]:
def ajustar_intensidade_rgb(rgb, k):
    """
    Altera a intensidade da imagem multiplicando diretamente os canais RGB por k (Slide 38).
    s_i = k * r_i
    """
    return np.clip(rgb * k, 0.0, 1.0)

In [ ]:
def ajustar_intensidade_hsi(hsi, k):
    """
    Altera a intensidade da imagem alterando somente o canal I no espaço HSI (Slide 38).
    Isso preserva as cores (Matiz e Saturação) sem causar distorções cromáticas.
    """
    hsi_mod = hsi.copy()
    hsi_mod[:, :, 2] = np.clip(hsi_mod[:, :, 2] * k, 0.0, 1.0)
    return hsi_to_rgb(hsi_mod)

# PROCESSAMENTO ESPACIAL

In [ ]:
def suavizar_canais(rgb, tamanho_filtro=5):
    """
    Aplica filtro de média separadamente em cada canal de uma imagem RGB (Slide 44).
    """
    kernel = np.ones((tamanho_filtro, tamanho_filtro), np.float32) / (tamanho_filtro ** 2)
    suavizada = np.zeros_like(rgb)
    for c in range(3):
        suavizada[:, :, c] = cv2.filter2D(rgb[:, :, c], -1, kernel)
    return suavizada

In [ ]:
def agucar_laplaciano(rgb, c=-1.0):
    """
    Aplica aguçamento usando a máscara do Laplaciano em cada canal RGB individualmente (Slide 46).
    """
    # Máscara do Laplaciano padrão
    kernel_lap = np.array([[0,  1, 0],
                           [1, -4, 1],
                           [0,  1, 0]], dtype=np.float32)

    agucada = np.zeros_like(rgb)
    for channel in range(3):
        img_c = rgb[:, :, channel]
        # Calcula o Laplaciano
        lap = cv2.filter2D(img_c, -1, kernel_lap)
        # Combina com a imagem original
        agucada[:, :, channel] = np.clip(img_c + c * lap, 0.0, 1.0)
    return agucada

# GERADOR DE IMAGEM DE TESTE (DIDÁTICO)

In [ ]:
def gerar_imagem_exemplo():
    h, w = 300, 300
    img = np.zeros((h, w, 3), dtype=np.float32)

    # Círculo vermelho (Saturação máxima)
    cv2.circle(img, (80, 80), 60, (1.0, 0.0, 0.0), -1)
    # Retângulo verde
    cv2.rectangle(img, (150, 20), (270, 140), (0.0, 1.0, 0.0), -1)
    # Elipse azul
    cv2.ellipse(img, (200, 220), (70, 40), 30, 0, 360, (0.0, 0.0, 1.0), -1)
    # Gradiente de cinza (para testar fatiamento de intensidade)
    for y in range(130, 210):
        val = (y - 130) / 80.0
        cv2.line(img, (30, y), (130, y), (val, val, val), 1)

    return img

In [ ]:
!mkdir resultados

# EXECUÇÃO DA DEMONSTRAÇÃO

In [ ]:
#if __name__ == "__main__":
def executar_processamento():
    #print("[1/5] Gerando imagem de teste sintética...")
    print("[1/5] CARREGANDO IMAGEM DE TESTE ...\n")
    img = cv2.cvtColor(cv2.imread("./pdsi/imagens/baboon.png", cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB).astype(np.float64) / 255.0

    print("[2/5] REALIZANDO DECOMPOSIÇÃO HSI ...")
    hsi = rgb_to_hsi(img)

    # Salva decomposição HSI
    fig, axes = plt.subplots(1, 4, figsize=(10, 2.5))
    axes[0].imshow(img)
    axes[0].set_title("Original RGB")
    axes[0].axis('off')

    im1 = axes[1].imshow(hsi[:, :, 0], cmap='hsv')
    axes[1].set_title("Matiz (Hue)")
    axes[1].axis('off')
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    im2 = axes[2].imshow(hsi[:, :, 1], cmap='gray')
    axes[2].set_title("Saturação (S)")
    axes[2].axis('off')
    plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

    im3 = axes[3].imshow(hsi[:, :, 2], cmap='gray')
    axes[3].set_title("Intensidade (I)")
    axes[3].axis('off')
    plt.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.savefig("./resultados/hsi_decomposition.png", dpi=150)
    plt.show()
    #plt.close()

    print("[3/5] APLICANDO FATIAMENTO DE PSEUDO-CORES ...\n")
    gray_img = hsi[:, :, 2]  # Usando o canal de intensidade
    limiares = [0.25, 0.5, 0.75]
    cores = [
        [0.0, 0.0, 0.6],  # Azul escuro (nível baixo de cinza)
        [0.0, 0.8, 0.8],  # Ciano
        [0.8, 0.8, 0.0],  # Amarelo
        [0.9, 0.0, 0.0]   # Vermelho brilhante (nível alto de cinza)
    ]
    img_sliced = fatiamento_intensidade(gray_img, limiares, cores)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(gray_img, cmap='gray')
    axes[0].set_title("Original Cinza (Intensidade)")
    axes[0].axis('off')

    axes[1].imshow(img_sliced)
    axes[1].set_title("Pseudo-cores (Fatiamento)")
    axes[1].axis('off')

    plt.tight_layout()
    plt.savefig("./resultados/pseudo_color_slicing.png", dpi=150)
    plt.show()
    #plt.close()

    print("[4/5] EXECUTANDO TRANSFORMAÇÕES DE CORES (COMPLEMENTO E AJUSTE DE BRILHO) ...\n")
    comp = complemento_cor(img)
    ajustada_rgb = ajustar_intensidade_rgb(img, 0.5)
    ajustada_hsi = ajustar_intensidade_hsi(hsi, 0.5)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(img)
    axes[0].set_title("Original")
    axes[0].axis('off')

    axes[1].imshow(comp)
    axes[1].set_title("Complemento")
    axes[1].axis('off')

    axes[2].imshow(ajustada_rgb)
    axes[2].set_title("Brilho Reduzido em RGB")
    axes[2].axis('off')

    axes[3].imshow(ajustada_hsi)
    axes[3].set_title("Brilho Reduzido em HSI")
    axes[3].axis('off')

    plt.tight_layout()
    plt.savefig("./resultados/color_transformations.png", dpi=150)
    plt.show()
    #plt.close()

    print("[5/5] EXECUTANDO PROCESSAMENTO ESPACIAL ...\n")
    suave = suavizar_canais(img, tamanho_filtro=9)
    aguda = agucar_laplaciano(img)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img)
    axes[0].set_title("Original")
    axes[0].axis('off')

    axes[1].imshow(suave)
    axes[1].set_title("Suavizada (Filtro Média)")
    axes[1].axis('off')

    axes[2].imshow(aguda)
    axes[2].set_title("Aguçada (Laplaciano)")
    axes[2].axis('off')

    plt.tight_layout()
    plt.savefig("./resultados/spatial_filtering.png", dpi=150)
    plt.show()
    #plt.close()

    print("### PROCESSAMENTO CONCLUÍDO COM SUCESSO! ###")

In [ ]:
executar_processamento()